# tensor-reshape-view — worked example 3: Flatten conv features with reshape for a linear layer

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-reshape-view`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

After a convolutional layer the feature map has shape `(B, C, H, W)`. Before passing it to a `nn.Linear` layer, you must collapse the spatial and channel dimensions into a single vector per sample. The standard idiom is `x.reshape(B, -1)` or `x.view(B, -1)`, which gives shape `(B, C*H*W)`.

## Worked solution

**Step 1 — Identify the dimensions.**
A batch of `B=4` images after convolution might have shape `(4, 16, 6, 6)`. The linear layer expects each sample as a 1-D vector, so we need shape `(4, 576)` where `576 = 16 * 6 * 6`.

**Step 2 — Use -1 to infer the flat size.**
`x.reshape(B, -1)` keeps the batch dimension and collapses everything else. PyTorch infers the second dimension from `numel() / B`.

**Step 3 — Why reshape and not view here.**
After common operations like `F.relu` on a contiguous tensor, the output is still contiguous, so `.view(B, -1)` also works. But `.reshape` is safer in general because it works even if an upstream operation returned a non-contiguous output.

**Step 4 — Verify shape before nn.Linear.**
Assert that `x.shape == (B, expected_features)` before passing to the linear layer to catch mismatches early.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(1)
B = 4
# Simulate conv output: (B, channels, h, w)
conv_out = t.randn(B, 16, 6, 6)
print('conv output shape:', conv_out.shape)  # (4, 16, 6, 6)

# Flatten spatial + channel dims for linear layer
flat = conv_out.reshape(B, -1)
print('flat shape:', flat.shape)             # (4, 576)
print('numel per sample:', flat.shape[1])    # 576 = 16*6*6

# Pass to a linear layer
linear = nn.Linear(576, 10)
out = linear(flat)
print('linear output shape:', out.shape)     # (4, 10)

# Also works with view (conv_out is contiguous)
flat_v = conv_out.view(B, -1)
print('view matches reshape:', t.allclose(flat, flat_v))  # True